# Defense-RL — GRPO Fine-Tuning on Colab T4

This notebook:
1. Clones your Defense-RL code from HuggingFace
2. Installs dependencies
3. Runs GRPO fine-tuning on Qwen2.5-1.5B-Instruct
4. Saves LoRA adapters back to HuggingFace Hub

**Runtime**: Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Cell 2: Clone your Space repo ─────────────────────────────────
!git clone https://huggingface.co/spaces/Bhaskar111/defense-ai defense-rl
%cd defense-rl
!ls

In [ ]:
# ── Cell 3: Install dependencies (takes ~3 min) ───────────────────
!pip install -q \
    transformers>=4.45.0 \
    trl>=0.11.0 \
    peft>=0.12.0 \
    accelerate>=0.30.0 \
    datasets>=2.20.0 \
    bitsandbytes>=0.43.0 \
    huggingface_hub \
    torch --quiet

In [ ]:
# ── Cell 4: Set your HF token (for saving checkpoints) ────────────
import os
from google.colab import userdata

# Option A: Use Colab Secrets (recommended)
# Go to: Secrets (key icon on left) → Add → Name: HF_TOKEN, Value: hf_xxx
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('Token loaded from Colab Secrets')
except:
    # Option B: Paste directly (less secure)
    HF_TOKEN = 'hf_YOUR_TOKEN_HERE'  # replace this
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('Token set manually')

In [ ]:
# ── Cell 5: Dry run first (verify everything works) ───────────────
!python train.py \
    --backend rule \
    --episodes 30 \
    --dry-run

In [ ]:
# ── Cell 6: FULL GRPO Training ────────────────────────────────────
# This fine-tunes Qwen2.5-1.5B-Instruct with GRPO on T4 GPU
# Expected time: ~30-60 min for 50 episodes, 3 epochs

!python train.py \
    --backend local \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --episodes 50 \
    --epochs 3 \
    --batch-size 1 \
    --lora-rank 16 \
    --device cuda \
    --checkpoint-dir checkpoints

In [ ]:
# ── Cell 7: Save trained adapters to HuggingFace Hub ─────────────
from huggingface_hub import HfApi, create_repo
import os

api = HfApi(token=os.environ['HF_TOKEN'])
USERNAME = 'Bhaskar111'  # your HF username

# Create model repos and upload adapters
for agent in ['radar', 'actor']:
    repo_id = f'{USERNAME}/defense-rl-{agent}-adapter'
    checkpoint_path = f'checkpoints/{agent}'
    
    if os.path.exists(checkpoint_path):
        create_repo(repo_id, repo_type='model', exist_ok=True, token=os.environ['HF_TOKEN'])
        api.upload_folder(
            folder_path=checkpoint_path,
            repo_id=repo_id,
            repo_type='model',
            token=os.environ['HF_TOKEN']
        )
        print(f'Uploaded {agent} adapter → https://huggingface.co/{repo_id}')
    else:
        print(f'No checkpoint found at {checkpoint_path}')

In [ ]:
# ── Cell 8: Quick evaluation after training ───────────────────────
!python train.py \
    --backend local \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --episodes 30 \
    --epochs 1 \
    --dry-run \
    --resume checkpoints